In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

current = Path.cwd()
ROOT = next(
    (
        folder
        for folder in [current, *current.parents]
        if (folder / "reports/inclusion_ledger_v1.csv").exists()
    ),
    None,
)

if ROOT is None:
    raise FileNotFoundError("Project root was not found.")

ledger = pd.read_csv(
    ROOT / "reports/inclusion_ledger_v1.csv",
    keep_default_na=False,
)

unknown = ledger.loc[
    ledger["inclusion_status"].eq("unknown")
].copy()

unknown_ids = pd.to_numeric(
    unknown["execution_number"], errors="raise"
).astype(int)

print("Records requiring upstream evidence:", len(unknown_ids))
print("Execution numbers:", sorted(unknown_ids.tolist()))

upstream_files = [
    "statements_extracted.csv",
    "statements_clean_v1.csv",
    "priority_review_queue.csv",
]

for filename in upstream_files:
    path = ROOT / "data/processed" / filename
    print(f"\n=== {filename} ===")

    if not path.exists():
        print("File not found.")
        continue

    table = pd.read_csv(path, keep_default_na=False)
    print("Available columns:", table.columns.tolist())

    if "execution_number" not in table.columns:
        print("No execution_number column; manual inspection required.")
        continue

    execution_ids = pd.to_numeric(
        table["execution_number"], errors="coerce"
    )
    selected = table.loc[execution_ids.isin(unknown_ids)].copy()

    preferred_columns = [
        "execution_number",
        "source_url",
        "extraction_status",
        "content_status",
        "cleaning_status",
        "error",
        "statement_text",
        "statement_section_raw",
    ]
    columns = [
        column
        for column in preferred_columns
        if column in selected.columns
    ]

    print("Matching records:", len(selected))

    with pd.option_context(
        "display.max_rows", 30,
        "display.max_colwidth", 180,
    ):
        display(selected[columns])

Records requiring upstream evidence: 21
Execution numbers: [48, 50, 135, 241, 387, 462, 463, 490, 509, 510, 512, 513, 533, 539, 552, 555, 567, 570, 573, 579, 601]

=== statements_extracted.csv ===
Available columns: ['execution_number', 'index_date', 'statement_page_date', 'date_check', 'source_url', 'html_file', 'content_status', 'statement_section_raw', 'statement_text_candidate', 'word_count', 'has_media_heading', 'has_spoken_heading', 'has_written_heading', 'has_spanish_marker', 'has_possible_editorial_note', 'manual_review_status', 'error']
Matching records: 21


,execution_number,source_url,content_status,error,statement_section_raw
2,601,https://www.tdcj.texas.gov/death_row/dr_info/normanlejameslast.html,no_statement_reported,,No last statement provided.
24,579,https://www.tdcj.texas.gov/death_row/dr_info/frattarobertlast.html,statement_candidate,,No last statement given.
30,573,https://www.tdcj.texas.gov/death_row/dr_info/rhoadesricklast.html,statement_candidate,,No statement was made.
33,570,https://www.tdcj.texas.gov/death_row/dr_info/wardlowbillylast.html,statement_candidate,,No statement given.
36,567,https://www.tdcj.texas.gov/death_row/dr_info/runnelstravislast.html,statement_candidate,,None
48,555,https://www.tdcj.texas.gov/death_row/dr_info/ackerdaniellast.html,statement_candidate,,No statement given.
51,552,https://www.tdcj.texas.gov/death_row/dr_info/bibledannylast.html,statement_candidate,,No last statement
64,539,https://www.tdcj.texas.gov/death_row/dr_info/wilkinschristopherlast.html,statement_candidate,,None
70,533,https://www.tdcj.texas.gov/death_row/dr_info/freemanjamesllast.html,statement_candidate,,None.
90,513,https://www.tdcj.texas.gov/death_row/dr_info/sellstommylast.html,statement_candidate,,This inmate declined to make a last statement.



=== statements_clean_v1.csv ===
Available columns: ['execution_number', 'index_date', 'statement_page_date', 'date_check', 'source_url', 'html_file', 'content_status', 'statement_section_raw', 'statement_text_candidate', 'word_count', 'has_media_heading', 'has_spoken_heading', 'has_written_heading', 'has_spanish_marker', 'has_possible_editorial_note', 'manual_review_status', 'error', 'cleaning_status', 'cleaning_note', 'analysis_text_v1']
Matching records: 21


,execution_number,source_url,content_status,cleaning_status,error,statement_section_raw
2,601,https://www.tdcj.texas.gov/death_row/dr_info/normanlejameslast.html,no_statement_reported,no_statement_reported,,No last statement provided.
24,579,https://www.tdcj.texas.gov/death_row/dr_info/frattarobertlast.html,statement_candidate,no_statement_reported,,No last statement given.
30,573,https://www.tdcj.texas.gov/death_row/dr_info/rhoadesricklast.html,statement_candidate,no_statement_reported,,No statement was made.
33,570,https://www.tdcj.texas.gov/death_row/dr_info/wardlowbillylast.html,statement_candidate,no_statement_reported,,No statement given.
36,567,https://www.tdcj.texas.gov/death_row/dr_info/runnelstravislast.html,statement_candidate,ambiguous_short_response,,None
48,555,https://www.tdcj.texas.gov/death_row/dr_info/ackerdaniellast.html,statement_candidate,no_statement_reported,,No statement given.
51,552,https://www.tdcj.texas.gov/death_row/dr_info/bibledannylast.html,statement_candidate,no_statement_reported,,No last statement
64,539,https://www.tdcj.texas.gov/death_row/dr_info/wilkinschristopherlast.html,statement_candidate,ambiguous_short_response,,None
70,533,https://www.tdcj.texas.gov/death_row/dr_info/freemanjamesllast.html,statement_candidate,ambiguous_short_response,,None.
90,513,https://www.tdcj.texas.gov/death_row/dr_info/sellstommylast.html,statement_candidate,no_statement_reported,,This inmate declined to make a last statement.



=== priority_review_queue.csv ===
Available columns: ['execution_number', 'index_date', 'statement_page_date', 'date_check', 'source_url', 'html_file', 'content_status', 'statement_section_raw', 'statement_text_candidate', 'word_count', 'has_media_heading', 'has_spoken_heading', 'has_written_heading', 'has_spanish_marker', 'has_possible_editorial_note', 'manual_review_status', 'error']
Matching records: 21


,execution_number,source_url,content_status,error,statement_section_raw
0,601,https://www.tdcj.texas.gov/death_row/dr_info/normanlejameslast.html,no_statement_reported,,No last statement provided.
2,579,https://www.tdcj.texas.gov/death_row/dr_info/frattarobertlast.html,statement_candidate,,No last statement given.
4,573,https://www.tdcj.texas.gov/death_row/dr_info/rhoadesricklast.html,statement_candidate,,No statement was made.
5,570,https://www.tdcj.texas.gov/death_row/dr_info/wardlowbillylast.html,statement_candidate,,No statement given.
6,567,https://www.tdcj.texas.gov/death_row/dr_info/runnelstravislast.html,statement_candidate,,None
9,555,https://www.tdcj.texas.gov/death_row/dr_info/ackerdaniellast.html,statement_candidate,,No statement given.
10,552,https://www.tdcj.texas.gov/death_row/dr_info/bibledannylast.html,statement_candidate,,No last statement
12,539,https://www.tdcj.texas.gov/death_row/dr_info/wilkinschristopherlast.html,statement_candidate,,None
13,533,https://www.tdcj.texas.gov/death_row/dr_info/freemanjamesllast.html,statement_candidate,,None.
16,513,https://www.tdcj.texas.gov/death_row/dr_info/sellstommylast.html,statement_candidate,,This inmate declined to make a last statement.


In [2]:
import hashlib

source = ROOT / "data/processed/statements_clean_v1.csv"
clean = pd.read_csv(source, keep_default_na=False)

required = [
    "execution_number",
    "source_url",
    "cleaning_status",
    "cleaning_note",
    "statement_section_raw",
]
missing = set(required) - set(clean.columns)
if missing:
    raise ValueError(f"Missing columns: {sorted(missing)}")

ids = pd.to_numeric(clean["execution_number"], errors="raise")
evidence = clean.loc[ids.isin(unknown_ids), required].copy()
evidence["execution_number"] = pd.to_numeric(
    evidence["execution_number"], errors="raise"
).astype(int)

if (
    len(evidence) != 21
    or not evidence["execution_number"].is_unique
    or set(evidence["execution_number"]) != set(unknown_ids)
):
    raise ValueError("Evidence records do not match the 21 unresolved IDs.")

evidence["evidence_file"] = "data/processed/statements_clean_v1.csv"
evidence["evidence_file_sha256"] = hashlib.sha256(
    source.read_bytes()
).hexdigest()
evidence["review_scope"] = "Upstream cleaning evidence; not a new webpage audit"

evidence = evidence.sort_values("execution_number")
target = ROOT / "reports/inclusion_evidence_21_v1.csv"

if target.exists():
    existing = pd.read_csv(target, keep_default_na=False)
    pd.testing.assert_frame_equal(
        existing,
        evidence.reset_index(drop=True),
        check_dtype=False,
    )
    print("Existing evidence file matches; nothing overwritten.")
else:
    evidence.to_csv(target, index=False, encoding="utf-8-sig")
    print("Created:", target.relative_to(ROOT))

display(
    evidence["cleaning_status"]
    .value_counts()
    .to_frame("records")
)
display(
    evidence[
        ["execution_number", "cleaning_status", "cleaning_note"]
    ]
)

Created: reports/inclusion_evidence_21_v1.csv


,records
cleaning_status,
no_statement_reported,13
summary_or_reference,4
ambiguous_short_response,4


,execution_number,cleaning_status,cleaning_note
555,48,summary_or_reference,Recorder summary of thanks
553,50,summary_or_reference,Poem title/reference; full recited text not su...
468,135,no_statement_reported,
362,241,summary_or_reference,Recorder reports an Irish utterance and its tr...
216,387,summary_or_reference,Recorder summary of profanity
141,462,no_statement_reported,
140,463,ambiguous_short_response,Inspect source context before classifying
113,490,no_statement_reported,
94,509,no_statement_reported,
93,510,no_statement_reported,


In [3]:
from bs4 import BeautifulSoup

review_ids = [241, 463, 533, 539, 567]

table = pd.read_csv(
    ROOT / "data/processed/statements_clean_v1.csv",
    keep_default_na=False,
)
numbers = pd.to_numeric(table["execution_number"], errors="raise")

for number in review_ids:
    row = table.loc[numbers.eq(number)].iloc[0]

    print("\n" + "=" * 70)
    print("Execution:", number)
    print("Source:", row["source_url"])
    print("Cleaning note:", row["cleaning_note"])
    print("Extracted section:", repr(row["statement_section_raw"]))

    recorded_path = Path(row["html_file"])
    candidates = (
        [recorded_path]
        if recorded_path.is_absolute()
        else [
            ROOT / recorded_path,
            ROOT / "data/raw" / recorded_path,
            ROOT / "data/raw" / recorded_path.name,
        ]
    )
    html_path = next(
        (path for path in candidates if path.is_file()),
        None,
    )

    if html_path is None:
        print("Cached HTML not found.")
        print("Recorded HTML path:", row["html_file"])
        continue

    soup = BeautifulSoup(
        html_path.read_bytes(),
        "html.parser",
    )
    for element in soup(["script", "style", "nav", "footer"]):
        element.decompose()

    content = soup.find("main") or soup.body or soup
    print("\nCached page text:\n")
    print(content.get_text("\n", strip=True))


Execution: 241
Source: https://www.tdcj.texas.gov/death_row/dr_info/goodwinalvinlast.html
Cleaning note: Recorder reports an Irish utterance and its translation
Extracted section: 'He spoke in Irish, translating to "Goodbye."'

Cached page text:

skip to main content
ALERT:  XXXXX
NOTICE:  XXXXX
ANNOUNCEMENT:  XXXXX
Home
|
Search
TEXAS DEPARTMENT OF CRIMINAL JUSTICE
Death Row Links
Last Statement
Date of Execution:
January 18, 2001
Inmate:
Alvin Goodwin #879
Last Statement:
He spoke in Irish, translating to "Goodbye."
Compact With Texans
Court Resources
Public Information Act
Report Waste, Fraud, and Abuse
Second-Chance Employers
Site Policies
State Energy Savings Program
TDCJ Intranet
Texas Correctional Industries
Texas Homeland Security
Texas Veterans Portal
TexasOnline
TRAIL Statewide Search
Website for Work
Texas Department of Criminal Justice
PO Box 99
Huntsville, Texas 77342-0099
(936) 295-6371
Facebook
Instagram
Twitter/X
YouTube
LinkedIn
© Texas Department of Criminal Justice.

In [4]:
import pandas as pd

ledger_path = ROOT / "reports/inclusion_ledger_v1.csv"
evidence_path = ROOT / "reports/inclusion_evidence_21_v1.csv"
target = ROOT / "reports/inclusion_ledger_v2.csv"

ledger = pd.read_csv(ledger_path, keep_default_na=False)
evidence = pd.read_csv(evidence_path, keep_default_na=False)

if not evidence["execution_number"].is_unique:
    raise ValueError("Duplicate evidence IDs.")

expected = set(
    ledger.loc[
        ledger["inclusion_status"].eq("unknown"),
        "execution_number",
    ].astype(int)
)
observed = set(evidence["execution_number"].astype(int))

if expected != observed or len(observed) != 21:
    raise ValueError("Evidence does not match the unresolved records.")

updated = ledger.copy()
updated["boundary_review_status"] = ""
updated["boundary_review_note"] = ""
updated["evidence_file_sha256"] = ""

reason_map = {
    "no_statement_reported": "no_statement_reported",
    "summary_or_reference": "summary_or_reference",
    "ambiguous_short_response": "ambiguous_short_response",
}

for _, record in evidence.iterrows():
    number = int(record["execution_number"])
    mask = updated["execution_number"].astype(int).eq(number)
    status = record["cleaning_status"]

    if status not in reason_map:
        raise ValueError(f"Unexpected cleaning status: {status}")

    updated.loc[mask, "inclusion_status"] = "excluded"
    updated.loc[mask, "exclusion_reason"] = reason_map[status]
    updated.loc[mask, "reason_evidence"] = (
        "reports/inclusion_evidence_21_v1.csv"
        f":execution_number={number}"
    )
    updated.loc[mask, "evidence_file_sha256"] = record[
        "evidence_file_sha256"
    ]
    updated.loc[mask, "evidence_limit"] = (
        "Historical exclusion documented from upstream cleaning evidence; "
        "not a blanket endorsement of the exclusion rule."
    )

    if status == "ambiguous_short_response":
        updated.loc[mask, "boundary_review_status"] = (
            "attribution_unresolved"
        )
        updated.loc[mask, "boundary_review_note"] = (
            "User-supplied cached page text contains only No/None; "
            "speaker attribution remains unresolved."
        )

    if number == 241:
        updated.loc[mask, "boundary_review_status"] = (
            "future_inclusion_candidate"
        )
        updated.loc[mask, "boundary_review_note"] = (
            "Cached page reports the explicit translation Goodbye. "
            "Original-language wording is unavailable. Review against "
            "the treatment of other supplied translations in a future version."
        )

updated = updated.reset_index(drop=True)

if target.exists():
    existing = pd.read_csv(target, keep_default_na=False)
    pd.testing.assert_frame_equal(
        existing, updated, check_dtype=False
    )
    print("Existing v2 ledger matches; nothing overwritten.")
else:
    updated.to_csv(target, index=False, encoding="utf-8-sig")
    print("Created:", target.relative_to(ROOT))

print("\nCurrent candidate-corpus inclusion status:")
display(updated["inclusion_status"].value_counts().to_frame("records"))

print("\nRecorded exclusion reasons:")
display(
    updated.loc[
        updated["inclusion_status"].eq("excluded"),
        "exclusion_reason",
    ].value_counts().to_frame("records")
)

print("\nFrozen corpus and model results were not modified.")

Created: reports/inclusion_ledger_v2.csv

Current candidate-corpus inclusion status:


,records
inclusion_status,
included,480
excluded,123



Recorded exclusion reasons:


,records
exclusion_reason,
shared_notice_link,102
no_statement_reported,13
ambiguous_short_response,4
summary_or_reference,4



Frozen corpus and model results were not modified.
